# All-Model Inference Evaluation (from best `.pth` checkpoints)

This notebook runs **fresh inference** for each model checkpoint and computes metrics directly from predictions.

## What this notebook does
- Finds the best available checkpoint (`*.pth`) for each model family
- Runs inference on BraTS processed test slices
- Computes sample-level metrics (AUROC, AUPRC, Accuracy, Precision, Recall, Specificity, F1, FPR, FNR)
- Computes lesion-only localization metrics (Dice/IoU) if masks exist
- Saves unified results tables for all models

## Expected data
- Images: `.../processed/images/slice_*.npy`
- Labels: `.../processed/labels/label_*.npy`
- Masks (optional but recommended): `.../processed/masks/slice_*.npy`

## Outputs
- `evaluations/tables/all_models_inference_metrics.csv`
- `evaluations/tables/all_models_inference_metrics_ranked.csv`

In [ ]:
# Cell 2: Environment and imports
import os
import sys
import importlib.util
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# SSIM
try:
    from skimage.metrics import structural_similarity as ssim
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-image'])
    from skimage.metrics import structural_similarity as ssim

# Colab Drive (if available)
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception:
    pass

# Resolve project/repo roots robustly
if Path('/content/drive/MyDrive/symAD-ECNN').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN')
elif Path('C:/Users/rifad/symAD-ECNN').exists():
    PROJECT_ROOT = Path('C:/Users/rifad/symAD-ECNN')
else:
    PROJECT_ROOT = Path.cwd().resolve()

if Path('/content/symAD-ECNN').exists():
    REPO_ROOT = Path('/content/symAD-ECNN')
else:
    REPO_ROOT = PROJECT_ROOT

EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
ECNN_DIR = EVALS_DIR / 'ecnn_thresholding'
for p in [REPO_ROOT, EVALS_DIR, ECNN_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

OUT_DIR = PROJECT_ROOT / 'evaluations' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {device}')

In [ ]:
# GitHub sync cell (optional): clone/pull repository in Colab
REPO_URL = 'https://github.com/RifaDeen/symAD-ECNN.git'
COLAB_REPO = Path('/content/symAD-ECNN')

if Path('/content').exists():
    if not COLAB_REPO.exists():
        print('Cloning repository from GitHub...')
        subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_REPO)])
    else:
        print('Repository exists. Pulling latest changes...')
        subprocess.check_call(['git', '-C', str(COLAB_REPO), 'pull'])

    REPO_ROOT = COLAB_REPO
    EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
    ECNN_DIR = EVALS_DIR / 'ecnn_thresholding'
    for p in [REPO_ROOT, EVALS_DIR, ECNN_DIR]:
        if p.exists() and str(p) not in sys.path:
            sys.path.insert(0, str(p))

    print(f'Updated REPO_ROOT: {REPO_ROOT}')
else:
    print('Not running in Colab. Skipping GitHub clone/pull cell.')

In [ ]:
# Cell 3: Model definitions + checkpoint discovery
def ensure_e2cnn_installed():
    if importlib.util.find_spec('e2cnn') is not None:
        return
    print('Installing e2cnn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'e2cnn'])

try:
    ensure_e2cnn_installed()
    from ecnn_model_loader import get_model_for_inference
    HAS_ECNN = True
except Exception as e:
    HAS_ECNN = False
    print(f'ECNN loader unavailable: {e}')

class CNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 1, 3, 1, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 256, 8, 8)
        return self.decoder(dec)

class LargeCNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 512, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 512)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 3, 2, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 1, 3, 1, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 512, 8, 8)
        return self.decoder(dec)

class ResNetAutoencoder(nn.Module):
    def __init__(self, finetune_strategy='none'):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])

        if finetune_strategy == 'none':
            for p in self.encoder.parameters():
                p.requires_grad = False
        elif finetune_strategy == 'partial':
            for i, module in enumerate(self.encoder.children()):
                train = i >= 7
                for p in module.parameters():
                    p.requires_grad = train
        elif finetune_strategy == 'full':
            for p in self.encoder.parameters():
                p.requires_grad = True

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

def get_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt and isinstance(ckpt['model_state_dict'], dict):
            return ckpt['model_state_dict']
        if 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
            return ckpt['state_dict']
    return ckpt

CHECKPOINT_ROOTS = [
    PROJECT_ROOT / 'models' / 'saved_models',
    REPO_ROOT / 'models' / 'saved_models',
    Path('/content/drive/MyDrive/symAD-ECNN/models/saved_models')
]
CHECKPOINT_ROOTS = [p for p in CHECKPOINT_ROOTS if p.exists()]
print('Checkpoint roots:')
for p in CHECKPOINT_ROOTS:
    print('-', p)

MODEL_CONFIGS = [
    {'name': 'ECNN-AE (Optimized)', 'key': 'ecnn_opt', 'type': 'ecnn', 'subdirs': ['ecnn_optimized', '.'], 'patterns': ['ecnn_optimized_best.pth', '*ecnn*best*.pth', '*ecnn*.pth']},
    {'name': 'CNN-AE', 'key': 'cnn_base', 'type': 'cnn_base', 'subdirs': ['cnn_ae', '.'], 'patterns': ['cnn_best.pth', '*cnn*best*.pth', '*cnn*.pth']},
    {'name': 'CNN-AE Augmented', 'key': 'cnn_aug', 'type': 'cnn_base', 'subdirs': ['cnn_ae_augmented', '.'], 'patterns': ['*cnn*aug*best*.pth', '*aug*cnn*.pth']},
    {'name': 'CNN-AE Large', 'key': 'cnn_large', 'type': 'cnn_large', 'subdirs': ['cnn_ae_large', '.'], 'patterns': ['*cnn*large*best*.pth', '*large*cnn*.pth']},
    {'name': 'ResNet-AE (frozen)', 'key': 'resnet_ae', 'type': 'resnet_none', 'subdirs': ['resnet_ae', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
    {'name': 'ResNet-AE (partial FT)', 'key': 'resnet_ft_partial', 'type': 'resnet_partial', 'subdirs': ['resnet_finetuned_partial', 'resnet_finetuned', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
    {'name': 'ResNet-AE (full FT)', 'key': 'resnet_ft_full', 'type': 'resnet_full', 'subdirs': ['resnet_finetuned_full', 'resnet_finetuned', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
]

def find_best_checkpoint(cfg):
    cands = []
    for root in CHECKPOINT_ROOTS:
        for sd in cfg['subdirs']:
            base = root if sd == '.' else root / sd
            if not base.exists():
                continue
            for pat in cfg['patterns']:
                cands.extend([p for p in base.rglob(pat) if p.is_file()])
    cands = sorted(set(cands))
    if not cands:
        return None
    # prioritize files with 'best' then newest modified
    cands = sorted(cands, key=lambda p: (('best' not in p.name.lower()), -p.stat().st_mtime))
    return cands[0]

def load_model_from_cfg(cfg):
    ckpt_path = find_best_checkpoint(cfg)
    if ckpt_path is None:
        raise FileNotFoundError('No checkpoint found')

    if cfg['type'] == 'ecnn':
        if not HAS_ECNN:
            raise RuntimeError('ECNN loader unavailable')
        model, _ = get_model_for_inference(str(ckpt_path), str(device))
        return model.to(device).eval(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location=device)
    sd = get_state_dict(ckpt)

    if cfg['type'] == 'cnn_large':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 512
        model = LargeCNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'cnn_base':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 256
        model = CNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'resnet_none':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['type'] == 'resnet_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    elif cfg['type'] == 'resnet_full':
        model = ResNetAutoencoder(finetune_strategy='full')
    else:
        raise ValueError(f"Unknown type: {cfg['type']}")

    model.load_state_dict(sd, strict=False)
    return model.to(device).eval(), ckpt_path

In [ ]:
# Cell 4: Dataset and evaluation helpers
class EvalDataset(Dataset):
    def __init__(self, image_dir: Path, label_dir: Path, mask_dir: Path | None = None, mode='grayscale'):
        self.image_files = sorted(image_dir.glob('slice_*.npy'))
        self.label_dir = label_dir
        self.mask_dir = mask_dir if (mask_dir is not None and mask_dir.exists()) else None
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        sid = img_path.stem.split('_')[-1]
        lab_path = self.label_dir / f'label_{sid}.npy'

        img = np.load(img_path).astype(np.float32)
        label = int(np.load(lab_path)[0])

        gray = torch.from_numpy(img).float().unsqueeze(0)
        target = gray.clone()

        if self.mask_dir is not None:
            mpath = self.mask_dir / f'slice_{sid}.npy'
            if mpath.exists():
                mask = np.load(mpath).astype(np.uint8)
            else:
                mask = np.zeros_like(img, dtype=np.uint8)
        else:
            mask = np.zeros_like(img, dtype=np.uint8)

        if self.mode == 'resnet':
            img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            return inp, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

        return gray, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

def normalize01(x):
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def dice_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    return float((2.0 * inter) / (pred.sum() + gt.sum() + 1e-8))

def iou_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    union = ((pred + gt) > 0).sum()
    return float(inter / (union + 1e-8))

def threshold_from_normals(scores, labels, target_fpr=0.20):
    normal_scores = scores[labels == 0]
    if len(normal_scores) == 0:
        return float(np.percentile(scores, 80))
    return float(np.percentile(normal_scores, (1 - target_fpr) * 100))

def compute_cls_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)
    preds = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'auroc': float(roc_auc_score(labels, scores)) if np.unique(labels).size > 1 else np.nan,
        'auprc': float(average_precision_score(labels, scores)) if np.unique(labels).size > 1 else np.nan,
        'accuracy': float(accuracy_score(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'specificity': float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan,
        'f1_score': float(f1_score(labels, preds, zero_division=0)),
        'fpr': float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        'fnr': float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

def _topk_mean_np(arr_2d: np.ndarray, frac: float) -> np.ndarray:
    n = arr_2d.shape[1]
    k = max(1, int(n * frac))
    part = np.partition(arr_2d, n - k, axis=1)[:, n - k:]
    return part.mean(axis=1)

@torch.no_grad()
def infer_scores_and_lesion_metrics(model, dataloader, pixel_percentile=95):
    # Multiple slice-level scores for robust ranking
    score_store = {
        'mse_mean': [],
        'mae_mean': [],
        'mse_p95': [],
        'mse_p99': [],
        'mae_p95': [],
        'mae_p99': [],
        'mse_top1pct_mean': [],
        'mae_top1pct_mean': [],
    }

    ssim_scores, labels = [], []
    lesion_dice, lesion_iou = [], []
    lesion_count, non_lesion_count = 0, 0

    for x, target, lab, sid, mask in tqdm(dataloader, leave=False):
        x = x.to(device)
        target = target.to(device)
        mask = mask.to(device)

        recon = model(x)
        if recon.shape[-2:] != target.shape[-2:]:
            recon = F.interpolate(recon, size=target.shape[-2:], mode='bilinear', align_corners=False)

        err_abs = torch.abs(recon - target)
        err_sq = (recon - target) ** 2

        err_abs_np = err_abs.detach().cpu().numpy()[:, 0]  # [B,H,W]
        err_sq_np = err_sq.detach().cpu().numpy()[:, 0]    # [B,H,W]
        flat_abs = err_abs_np.reshape(err_abs_np.shape[0], -1)
        flat_sq = err_sq_np.reshape(err_sq_np.shape[0], -1)

        # Score candidates
        score_store['mse_mean'].extend(flat_sq.mean(axis=1).tolist())
        score_store['mae_mean'].extend(flat_abs.mean(axis=1).tolist())
        score_store['mse_p95'].extend(np.percentile(flat_sq, 95, axis=1).tolist())
        score_store['mse_p99'].extend(np.percentile(flat_sq, 99, axis=1).tolist())
        score_store['mae_p95'].extend(np.percentile(flat_abs, 95, axis=1).tolist())
        score_store['mae_p99'].extend(np.percentile(flat_abs, 99, axis=1).tolist())
        score_store['mse_top1pct_mean'].extend(_topk_mean_np(flat_sq, 0.01).tolist())
        score_store['mae_top1pct_mean'].extend(_topk_mean_np(flat_abs, 0.01).tolist())

        labels.extend(lab.numpy().tolist())

        # SSIM per sample
        t_np = target.detach().cpu().numpy()
        r_np = recon.detach().cpu().numpy()
        for b in range(t_np.shape[0]):
            gt_img = t_np[b, 0].astype(np.float32)
            rc_img = r_np[b, 0].astype(np.float32)
            dr = float(max(gt_img.max(), rc_img.max()) - min(gt_img.min(), rc_img.min()))
            dr = dr if dr > 1e-8 else 1.0
            ssim_scores.append(float(ssim(gt_img, rc_img, data_range=dr)))

        # lesion-only dice/iou from per-sample error maps
        err = err_abs.detach().cpu().numpy()  # [B,1,H,W]
        msk = (mask.detach().cpu().numpy() > 0).astype(np.uint8)

        for b in range(err.shape[0]):
            gt = msk[b, 0]
            if gt.sum() > 0:
                lesion_count += 1
                emap = normalize01(err[b, 0])
                thr = np.percentile(emap, pixel_percentile)
                pred = (emap >= thr).astype(np.uint8)
                lesion_dice.append(dice_score(pred, gt))
                lesion_iou.append(iou_score(pred, gt))
            else:
                non_lesion_count += 1

    labels = np.asarray(labels, dtype=np.int32)
    ssim_scores = np.asarray(ssim_scores, dtype=np.float32)
    score_store = {k: np.asarray(v, dtype=np.float32) for k, v in score_store.items()}

    # Backward-compatible names
    score_store['mse_scores'] = score_store['mse_mean']
    score_store['mae_scores'] = score_store['mae_mean']

    return {
        'scores': score_store['mse_mean'],  # default score
        'score_store': score_store,
        'mse_scores': score_store['mse_mean'],
        'mae_scores': score_store['mae_mean'],
        'ssim_scores': ssim_scores,
        'labels': labels,
        'lesion_dice': lesion_dice,
        'lesion_iou': lesion_iou,
        'lesion_count': int(lesion_count),
        'non_lesion_count': int(non_lesion_count),
    }

In [ ]:
# Cell 6: Run all-model inference and build metrics table
import zipfile

def has_required_dirs(p: Path) -> bool:
    return (p / 'images').exists() and (p / 'labels').exists()

# =========================
# DATA PATH RESOLUTION (matches your ecnn_ablation layout)
# =========================
BASE_EVAL_DATA_DIR = Path('/content/test_eval_data') if Path('/content').exists() else (PROJECT_ROOT / 'data' / 'test_eval_data')
DATA_DIR = BASE_EVAL_DATA_DIR / 'processed'
PROCESSED_ZIP = PROJECT_ROOT / 'data' / 'brats_test_with_masks_processed.zip'

IMAGE_DIR = DATA_DIR / 'images'
LABEL_DIR = DATA_DIR / 'labels'
MASK_DIR = DATA_DIR / 'masks'

initial_data_exists = IMAGE_DIR.exists() and LABEL_DIR.exists()

if not initial_data_exists and PROCESSED_ZIP.exists():
    print(f'Extracting processed zip: {PROCESSED_ZIP}')
    BASE_EVAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PROCESSED_ZIP, 'r') as zf:
        zf.extractall(BASE_EVAL_DATA_DIR)

    if (BASE_EVAL_DATA_DIR / 'processed' / 'images').exists() and (BASE_EVAL_DATA_DIR / 'processed' / 'labels').exists():
        DATA_DIR = BASE_EVAL_DATA_DIR / 'processed'
        print(f"Data found in 'processed' subdirectory: {DATA_DIR}")
    elif (BASE_EVAL_DATA_DIR / 'images').exists() and (BASE_EVAL_DATA_DIR / 'labels').exists():
        DATA_DIR = BASE_EVAL_DATA_DIR
        print(f"Data found directly in evaluation data directory: {DATA_DIR}")
    else:
        candidates = [BASE_EVAL_DATA_DIR] + [d for d in BASE_EVAL_DATA_DIR.iterdir() if d.is_dir()]
        DATA_DIR = next((p for p in candidates if has_required_dirs(p)), DATA_DIR)

    IMAGE_DIR = DATA_DIR / 'images'
    LABEL_DIR = DATA_DIR / 'labels'
    MASK_DIR = DATA_DIR / 'masks'

if not IMAGE_DIR.exists() or not LABEL_DIR.exists():
    raise FileNotFoundError(f'Processed data not found at {DATA_DIR}. Run test_preprocessing first.')

print(f'DATA_DIR: {DATA_DIR}')
print(f'Images: {len(list(IMAGE_DIR.glob("slice_*.npy")))}')
print(f'Labels: {len(list(LABEL_DIR.glob("label_*.npy")))}')
print(f'Masks available: {MASK_DIR.exists()}')

TARGET_FPR = 0.20
BATCH_SIZE = 32
PIXEL_PERCENTILE = 95

rows = []
roc_curves = []

for cfg in MODEL_CONFIGS:
    print(f"\n=== Evaluating {cfg['name']} ===")
    try:
        model, ckpt_path = load_model_from_cfg(cfg)
        mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'

        ds = EvalDataset(IMAGE_DIR, LABEL_DIR, MASK_DIR if MASK_DIR.exists() else None, mode=mode)
        dl = DataLoader(
            ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        out = infer_scores_and_lesion_metrics(model, dl, pixel_percentile=PIXEL_PERCENTILE)
        labels = out['labels']
        score_store = out['score_store']
        mse_scores = out['mse_scores']
        mae_scores = out['mae_scores']
        ssim_scores = out['ssim_scores']

        normal_mask = (labels == 0)
        lesion_mask = (labels == 1)

        # Evaluate all scoring choices; pick the best AUROC for robust slice-level detection
        candidate_scores = {
            'mse_mean': score_store['mse_mean'],
            'mae_mean': score_store['mae_mean'],
            'mse_p95': score_store['mse_p95'],
            'mse_p99': score_store['mse_p99'],
            'mae_p95': score_store['mae_p95'],
            'mae_p99': score_store['mae_p99'],
            'mse_top1pct_mean': score_store['mse_top1pct_mean'],
            'mae_top1pct_mean': score_store['mae_top1pct_mean'],
            '1_minus_ssim': (1.0 - ssim_scores),
        }

        score_stats = {}
        if np.unique(labels).size > 1:
            for k, v in candidate_scores.items():
                score_stats[k] = {
                    'auroc': float(roc_auc_score(labels, v)),
                    'auprc': float(average_precision_score(labels, v)),
                    'gap': float(np.mean(v[lesion_mask]) - np.mean(v[normal_mask])) if (normal_mask.any() and lesion_mask.any()) else np.nan,
                }
        else:
            for k, _ in candidate_scores.items():
                score_stats[k] = {'auroc': np.nan, 'auprc': np.nan, 'gap': np.nan}

        best_score_name = max(score_stats, key=lambda k: (score_stats[k]['auroc'] if score_stats[k]['auroc'] == score_stats[k]['auroc'] else -1.0))
        selected_scores = candidate_scores[best_score_name]
        selected_auroc = score_stats[best_score_name]['auroc']
        selected_auprc = score_stats[best_score_name]['auprc']
        selected_gap = score_stats[best_score_name]['gap']

        thr = threshold_from_normals(selected_scores, labels, TARGET_FPR)
        cls = compute_cls_metrics(labels, selected_scores, thr)

        fpr_vals, tpr_vals, _ = roc_curve(labels, selected_scores) if np.unique(labels).size > 1 else (np.array([0, 1]), np.array([0, 1]), None)
        roc_curves.append((cfg['name'], fpr_vals, tpr_vals, cls['auroc']))

        row = {
            'model_name': cfg['name'],
            'checkpoint': str(ckpt_path),
            'selected_score_name': best_score_name,
            'n_samples': int(len(labels)),
            'n_normal_labels': int(normal_mask.sum()),
            'n_lesion_labels': int(lesion_mask.sum()),
            'n_lesion_slices': int(out['lesion_count']),
            'n_non_lesion_slices': int(out['non_lesion_count']),

            # score separability diagnostics
            'mean_selected_score_normal': float(np.mean(selected_scores[normal_mask])) if normal_mask.any() else np.nan,
            'mean_selected_score_lesion': float(np.mean(selected_scores[lesion_mask])) if lesion_mask.any() else np.nan,
            'selected_score_gap_lesion_minus_normal': selected_gap,

            # all-candidate AUROC/AUPRC
            'auroc_mse_mean': score_stats['mse_mean']['auroc'],
            'auprc_mse_mean': score_stats['mse_mean']['auprc'],
            'auroc_mse_p95': score_stats['mse_p95']['auroc'],
            'auprc_mse_p95': score_stats['mse_p95']['auprc'],
            'auroc_mse_p99': score_stats['mse_p99']['auroc'],
            'auprc_mse_p99': score_stats['mse_p99']['auprc'],
            'auroc_mse_top1pct_mean': score_stats['mse_top1pct_mean']['auroc'],
            'auprc_mse_top1pct_mean': score_stats['mse_top1pct_mean']['auprc'],
            'auroc_mae_mean': score_stats['mae_mean']['auroc'],
            'auprc_mae_mean': score_stats['mae_mean']['auprc'],
            'auroc_1_minus_ssim': score_stats['1_minus_ssim']['auroc'],
            'auprc_1_minus_ssim': score_stats['1_minus_ssim']['auprc'],

            # reconstruction summaries
            'mean_mse_all': float(np.mean(mse_scores)) if len(mse_scores) else np.nan,
            'std_mse_all': float(np.std(mse_scores)) if len(mse_scores) else np.nan,
            'mean_mse_normal': float(np.mean(mse_scores[normal_mask])) if normal_mask.any() else np.nan,
            'mean_mse_lesion': float(np.mean(mse_scores[lesion_mask])) if lesion_mask.any() else np.nan,

            'mean_mae_all': float(np.mean(mae_scores)) if len(mae_scores) else np.nan,
            'std_mae_all': float(np.std(mae_scores)) if len(mae_scores) else np.nan,
            'mean_mae_normal': float(np.mean(mae_scores[normal_mask])) if normal_mask.any() else np.nan,
            'mean_mae_lesion': float(np.mean(mae_scores[lesion_mask])) if lesion_mask.any() else np.nan,

            'mean_ssim_all': float(np.mean(ssim_scores)) if len(ssim_scores) else np.nan,
            'std_ssim_all': float(np.std(ssim_scores)) if len(ssim_scores) else np.nan,
            'mean_ssim_normal': float(np.mean(ssim_scores[normal_mask])) if normal_mask.any() else np.nan,
            'mean_ssim_lesion': float(np.mean(ssim_scores[lesion_mask])) if lesion_mask.any() else np.nan,

            # lesion localization
            'mean_dice_lesion_only': float(np.mean(out['lesion_dice'])) if len(out['lesion_dice']) else np.nan,
            'std_dice_lesion_only': float(np.std(out['lesion_dice'])) if len(out['lesion_dice']) else np.nan,
            'mean_iou_lesion_only': float(np.mean(out['lesion_iou'])) if len(out['lesion_iou']) else np.nan,
            'pixel_percentile_for_mask': PIXEL_PERCENTILE,
            'threshold_method': f'normal-only FPR@{TARGET_FPR:.2f}',

            # final classification using selected score
            'selected_auroc': selected_auroc,
            'selected_auprc': selected_auprc,
            **cls,
        }
        rows.append(row)

        print(
            f"Selected score={best_score_name} | AUROC={selected_auroc:.4f} | AUPRC={selected_auprc:.4f} | "
            f"gap={selected_gap:.6f}"
        )
        print(
            f"TP={cls['tp']} TN={cls['tn']} FP={cls['fp']} FN={cls['fn']} | "
            f"MSE={row['mean_mse_all']:.6f} MAE={row['mean_mae_all']:.6f} SSIM={row['mean_ssim_all']:.4f}"
        )

    except Exception as e:
        print(f"FAILED: {cfg['name']} -> {e}")

results_df = pd.DataFrame(rows)
if results_df.empty:
    print('No models were evaluated successfully. Check checkpoint availability.')
else:
    results_df = results_df.sort_values('selected_auroc', ascending=False, na_position='last').reset_index(drop=True)
    display(results_df)

    raw_out = OUT_DIR / 'all_models_inference_metrics.csv'
    rank_out = OUT_DIR / 'all_models_inference_metrics_ranked.csv'
    results_df.to_csv(raw_out, index=False)
    results_df.assign(rank=np.arange(1, len(results_df) + 1)).to_csv(rank_out, index=False)

    print(f'\nSaved: {raw_out}')
    print(f'Saved: {rank_out}')

In [ ]:
# Cell 7: Compact final leaderboard
if 'results_df' in globals() and not results_df.empty:
    cols = [
        'model_name', 'selected_score_name',
        'n_samples', 'n_normal_labels', 'n_lesion_labels',
        'tp', 'tn', 'fp', 'fn',
        'selected_auroc', 'selected_auprc', 'accuracy', 'precision', 'recall', 'specificity', 'f1_score',
        'mean_selected_score_normal', 'mean_selected_score_lesion', 'selected_score_gap_lesion_minus_normal',
        'mean_mse_all', 'mean_mae_all', 'mean_ssim_all',
        'mean_dice_lesion_only', 'mean_iou_lesion_only',
        'n_lesion_slices', 'n_non_lesion_slices'
    ]
    cols = [c for c in cols if c in results_df.columns]
    leaderboard = results_df[cols].copy()
    leaderboard.insert(0, 'rank', np.arange(1, len(leaderboard)+1))
    display(leaderboard)
else:
    print('No leaderboard available.')

In [ ]:
# Cell 7: ROC comparison plot
if 'roc_curves' in globals() and len(roc_curves) > 0:
    fig, ax = plt.subplots(figsize=(8, 7))
    for name, fpr_vals, tpr_vals, auroc_val in roc_curves:
        label = f"{name} (AUROC={auroc_val:.3f})" if auroc_val == auroc_val else f"{name} (AUROC=nan)"
        ax.plot(fpr_vals, tpr_vals, lw=2, label=label)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('All-Model ROC Comparison (Inference-based)')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()

    fig_path = OUT_DIR / 'all_models_inference_roc_comparison.png'
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print(f'Saved: {fig_path}')
else:
    print('ROC curves not available.')